# SoccerNet GSR baseline on Colab (data from S3)

**Before running:**
1. Runtime > Change runtime type > **T4 GPU**.
2. Key icon in the left sidebar > add secrets `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `GITHUB_TOKEN`, and turn on **Notebook access** for each.
3. The PR adding `third_party/sn-gamestate` must be merged into `main`.

Then run the cells top to bottom. Skip the fallback cell unless the environment cell fails at the mmcv step.

In [1]:
# 1. Settings and secrets
import os
from google.colab import userdata

BUCKET = 'cmu-soccer-gsr-data'
S3_PREFIX = 'SoccerNetGS/gamestate-2024'   # folder holding train/valid/test/challenge .zip
AWS_REGION = 'us-east-2'                  # US East (Ohio)
SPLITS = ['valid']                        # add 'test' etc. only if needed (each zip is 5-10 GB)
REPO_URL = 'github.com/cmu-soccer-gsr/soccer-trajectory-prediction.git'

for k in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'GITHUB_TOKEN']:
    os.environ[k] = userdata.get(k)
os.environ['AWS_DEFAULT_REGION'] = AWS_REGION
os.environ['BUCKET'] = BUCKET
os.environ['S3_PREFIX'] = S3_PREFIX
os.environ['SPLITS'] = ' '.join(SPLITS)
os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_DIR'] = '/content/soccer-trajectory-prediction'
os.environ['DATA_DIR'] = '/content/soccer-trajectory-prediction/data'
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
print('Settings loaded.')

Settings loaded.


In [2]:
# 2. Check GPU and free disk (need roughly 25 GB free for the valid split)
!nvidia-smi
!df -h /content

Mon Oct 19 22:49:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%bash
# 3. Clone (or update) the team repo
set -e
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR" && git pull "https://${GITHUB_TOKEN}@${REPO_URL}" main
else
  git clone "https://${GITHUB_TOKEN}@${REPO_URL}" "$REPO_DIR"
  cd "$REPO_DIR" && git remote set-url origin "https://${REPO_URL}"   # keep token out of .git/config
fi
ls "$REPO_DIR/third_party/sn-gamestate"

ChallengeRules.md
examples_predictions
FAQ.md
images
LICENSE
plugins
poetry.lock
pretrained_models
pyproject.toml
README.md
sn_gamestate
uv.lock


Cloning into '/content/soccer-trajectory-prediction'...


In [6]:
%%bash
# 4. Build the baseline environment (10-20 min in a fresh session)
set -e

# Colab sets UV_SYSTEM_PYTHON=1, which makes uv install into Colab's own Python 3.13 instead of the venv
unset UV_SYSTEM_PYTHON UV_PYTHON

# Install uv if this session doesn't have it yet
command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:/usr/local/bin:$PATH"

cd "$REPO_DIR/third_party/sn-gamestate"

# Create the Python 3.9 environment (skipped if it already exists in this session)
if [ ! -x .venv/bin/python ]; then
  uv venv --python 3.9 --seed
fi
source .venv/bin/activate

# Baseline + TrackLab + torch 1.13.1
uv pip install --python .venv/bin/python -e .

# Newer setuptools removed pkg_resources, which mmcv/mmengine still import
uv pip install --python .venv/bin/python "setuptools<70"

# Prebuilt mmcv 2.0.1 wheel for torch 1.13.1 + CUDA 11.7 (no compiling)
uv pip install --python .venv/bin/python mmcv==2.0.1 \
  -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13/index.html

# Verify
python -c "import sys, torch, mmcv; print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available(), '| mmcv', mmcv.__version__)"
pip show tracklab | grep -i '^version'

python 3.9.25 | torch 1.13.1+cu117 | cuda 11.7 | gpu True | mmcv 2.0.1
Version: 1.3.24


Resolved 193 packages in 777ms
Prepared 1 package in 747ms
Uninstalled 1 package in 0.46ms
Installed 1 package in 0.80ms
 ~ sn-gamestate==0.2.0 (from file:///content/soccer-trajectory-prediction/third_party/sn-gamestate)
Resolved 1 package in 70ms
Prepared 1 package in 83ms
Uninstalled 1 package in 13ms
Installed 1 package in 5ms
 - setuptools==82.0.1
 + setuptools==69.5.1
Resolved 26 packages in 104ms
Prepared 4 packages in 4.27s
Installed 4 packages in 11ms
 + addict==2.4.0
 + mmcv==2.0.1
 + mmengine==0.10.7
 + termcolor==3.1.0


**Fallback (only if cell 4 failed at the mmcv step):** installs a torch/CUDA combination that has a prebuilt mmcv 2.0.1 wheel.

In [ ]:
%%bash
# 4b. FALLBACK - run only if cell 4 failed
set -e
cd "$REPO_DIR/third_party/sn-gamestate"
uv pip install torch==1.13.1 torchvision==0.14.1 --extra-index-url https://download.pytorch.org/whl/cu117
uv pip install mmcv==2.0.1 -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13/index.html
source .venv/bin/activate
python -c "import torch, mmcv; print('torch', torch.__version__, '| gpu', torch.cuda.is_available(), '| mmcv', mmcv.__version__)"

In [7]:
%%bash
# 5. Download and unzip the selected splits from S3 (zip is deleted after unzipping to save disk)
set -e
pip install -q awscli
mkdir -p "$DATA_DIR/SoccerNetGS/zips"
cd "$DATA_DIR/SoccerNetGS"
for s in $SPLITS; do
  if [ -d "$s" ] && [ -n "$(ls -A "$s")" ]; then echo "$s already present, skipping"; continue; fi
  echo "Downloading $s.zip ..."
  aws s3 cp "s3://$BUCKET/$S3_PREFIX/$s.zip" "zips/$s.zip"
  echo "Unzipping $s.zip ..."
  unzip -q "zips/$s.zip" -d "$s"
  rm "zips/$s.zip"
done
rmdir zips 2>/dev/null || true
for s in train valid test challenge; do mkdir -p "$s"; done   # empty placeholders for splits not downloaded
du -sh *
df -h /content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 46.6 MB/s eta 0:00:00
download: s3://cmu-soccer-gsr-data/SoccerNetGS/gamestate-2024/valid.zip to zips/valid.zip
Unzipping valid.zip ...
4.0K	challenge
4.0K	test
4.0K	train
12G	valid
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   60G  176G  26% /


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [8]:
# 6. Verify the data layout and dataset version (should be 1.3)
import json, glob, os
root = os.environ['DATA_DIR'] + '/SoccerNetGS/valid'
labels = sorted(glob.glob(root + '/*/Labels-GameState.json'))
print(len(labels), 'validation clips found')
if labels:
    print('example:', labels[0])
    print('dataset version:', json.load(open(labels[0]))['info']['version'])
else:
    print('No clips found directly under valid/. Nested folders found:')
    print(glob.glob(root + '/**/Labels-GameState.json', recursive=True)[:3])

58 validation clips found
example: /content/soccer-trajectory-prediction/data/SoccerNetGS/valid/SNGS-021/Labels-GameState.json
dataset version: 1.3


In [11]:
#Removes empty train, test, and challenge folders
%%bash
cd "$DATA_DIR/SoccerNetGS"
for s in train test challenge; do [ -d "$s" ] && [ -z "$(ls -A "$s")" ] && rmdir "$s" && echo "removed empty $s"; done; true
ls

removed empty train
removed empty test
removed empty challenge
valid


In [12]:
%%bash
# 7. Run the baseline (default: one validation clip; prints GS-HOTA at the end)
# To run more clips, add a dataset override such as dataset.nvid=3 (check soccernet.yaml for the exact key).
cd "$REPO_DIR/third_party/sn-gamestate"
source .venv/bin/activate
export MPLBACKEND=Agg   # Colab sets an inline backend this older matplotlib rejects
tracklab -cn soccernet data_dir="$DATA_DIR"

[10/19/26 23:12:23] INFO     Run directory:                          main.py:103
                             /content/soccer-trajectory-prediction/t            
                             hird_party/sn-gamestate/outputs/sn-game            
                             state/2026-10-19/23-12-23                          
                    INFO     Using device: 'cuda'.                   main.py:104
[10/19/26 23:12:26] WARNING  Warning: The 'train' set soccernet_game_state.py:36
                             does not exist in the                              
                             SoccerNetGS dataset at                             
                             '/content/soccer-traject                           
                             ory-prediction/data/Socc                           
                             erNetGS'. Please check                             
                             the path or download the                           
                            

Downloading: "https://download.openmmlab.com/mmocr/textdet/dbnet/dbnet_resnet18_fpnc_1200e_icdar2015/dbnet_resnet18_fpnc_1200e_icdar2015_20220825_221614-7c0e94f2.pth" to /root/.cache/torch/hub/checkpoints/dbnet_resnet18_fpnc_1200e_icdar2015_20220825_221614-7c0e94f2.pth
Downloading: "https://download.openmmlab.com/mmocr/textrecog/sar/sar_resnet31_parallel-decoder_5e_st-sub_mj-sub_sa_real/sar_resnet31_parallel-decoder_5e_st-sub_mj-sub_sa_real_20220915_171910-04eb4e75.pth" to /root/.cache/torch/hub/checkpoints/sar_resnet31_parallel-decoder_5e_st-sub_mj-sub_sa_real_20220915_171910-04eb4e75.pth
100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


In [14]:
%%bash
# 8a. Make a slimmed copy of the baseline tracker state (one time, ~5 min)
cd "$REPO_DIR/third_party/sn-gamestate"
mkdir -p states
[ -f states/baseline-valid.pklz ] || wget -q -O states/baseline-valid.pklz "https://zenodo.org/records/11065177/files/gamestate-prtreid-strongsort-valid-compressed.pklz?download=1"
source .venv/bin/activate
python - <<'PY'
import json, pickle, zipfile, gc
import pandas as pd
DROP = {"embeddings", "body_masks", "visibility_scores", "costs"}
src, dst = "states/baseline-valid.pklz", "states/baseline-valid-slim.pklz"
with zipfile.ZipFile(src) as zin, zipfile.ZipFile(dst, "w", zipfile.ZIP_DEFLATED) as zout:
    for name in zin.namelist():
        with zin.open(name, force_zip64=True) as f:
            if name == "summary.json":
                s = json.load(f)
                cols = s["columns"]
                if isinstance(cols, list):
                    s["columns"] = [c for c in cols if c not in DROP]
                else:
                    s["columns"] = {k: [c for c in v if c not in DROP] for k, v in cols.items()}
                zout.writestr(name, json.dumps(s, indent=2))
                continue
            df = pd.read_pickle(f)
        if not name.endswith("_image.pkl"):
            df = df.drop(columns=[c for c in DROP if c in df.columns])
        with zout.open(name, "w", force_zip64=True) as g:
            pickle.dump(df, g, protocol=pickle.DEFAULT_PROTOCOL)
        del df; gc.collect()
        print("slimmed", name, flush=True)
PY
ls -lh states/

slimmed 026.pkl
slimmed 026_image.pkl
slimmed 055.pkl
slimmed 055_image.pkl
slimmed 041.pkl
slimmed 041_image.pkl
slimmed 024.pkl
slimmed 024_image.pkl
slimmed 095.pkl
slimmed 095_image.pkl
slimmed 028.pkl
slimmed 028_image.pkl
slimmed 080.pkl
slimmed 080_image.pkl
slimmed 033.pkl
slimmed 033_image.pkl
slimmed 029.pkl
slimmed 029_image.pkl
slimmed 032.pkl
slimmed 032_image.pkl
slimmed 037.pkl
slimmed 037_image.pkl
slimmed 038.pkl
slimmed 038_image.pkl
slimmed 087.pkl
slimmed 087_image.pkl
slimmed 090.pkl
slimmed 090_image.pkl
slimmed 022.pkl
slimmed 022_image.pkl
slimmed 058.pkl
slimmed 058_image.pkl
slimmed 035.pkl
slimmed 035_image.pkl
slimmed 052.pkl
slimmed 052_image.pkl
slimmed 047.pkl
slimmed 047_image.pkl
slimmed 023.pkl
slimmed 023_image.pkl
slimmed 096.pkl
slimmed 096_image.pkl
slimmed 086.pkl
slimmed 086_image.pkl
slimmed 039.pkl
slimmed 039_image.pkl
slimmed 079.pkl
slimmed 079_image.pkl
slimmed 054.pkl
slimmed 054_image.pkl
slimmed 021.pkl
slimmed 021_image.pkl
slimmed 084.

In [15]:
%%bash
# 8. Score the published baseline tracker state on all validation clips (no models run)
cd "$REPO_DIR/third_party/sn-gamestate"
source .venv/bin/activate
export MPLBACKEND=Agg
free -h
tracklab -cn soccernet data_dir="$DATA_DIR" 'pipeline=[]' state.load_file="$(pwd)/states/baseline-valid-slim.pklz" state.save_file=null dataset.nvid=-1

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       8.9Gi       6.5Mi       3.0Gi        11Gi
Swap:             0B          0B          0B
[10/20/26 00:21:43] INFO     Run directory:                          main.py:103
                             /content/soccer-trajectory-prediction/t            
                             hird_party/sn-gamestate/outputs/sn-game            
                             state/2026-10-20/00-21-43                          
                    INFO     Using device: 'cuda'.                   main.py:104
[10/20/26 00:21:47] WARNING  Warning: The 'train' set soccernet_game_state.py:36
                             does not exist in the                              
                             SoccerNetGS dataset at                             
                             '/content/soccer-traject                           
                             ory-prediction/data/Socc           

100%|██████████| 58/58 [00:57<00:00,  1.01it/s]


In [16]:
# 9. Save run outputs to Google Drive before the session ends (/content is wiped on disconnect)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gsr-results
!cp -r "$REPO_DIR/third_party/sn-gamestate/outputs" /content/drive/MyDrive/gsr-results/
!ls /content/drive/MyDrive/gsr-results/outputs

Mounted at /content/drive
sn-gamestate


In [17]:
%%bash
CACHE=/content/drive/MyDrive/gsr-cache
mkdir -p $CACHE
cd /content/soccer-trajectory-prediction/third_party/sn-gamestate

# The Python 3.9 environment, plus the Python interpreter uv downloaded (the env depends on it)
tar -cf $CACHE/venv.tar .venv
tar -cf $CACHE/uv-python.tar -C /root/.local/share/uv python

# Model weights downloaded on the first run
[ -d pretrained_models ] && tar -cf $CACHE/weights.tar pretrained_models

# A small dev subset of validation clips (the first 5), so you don't need the full 10 GB
cd /content/soccer-trajectory-prediction/data/SoccerNetGS
mkdir -p $CACHE/valid5 && for v in $(ls valid | head -5); do cp -r valid/$v $CACHE/valid5/; done

du -sh $CACHE/*

83M	/content/drive/MyDrive/gsr-cache/uv-python.tar
699M	/content/drive/MyDrive/gsr-cache/valid5
5.0G	/content/drive/MyDrive/gsr-cache/venv.tar
1.1G	/content/drive/MyDrive/gsr-cache/weights.tar
